# Getting started with InstaNovo-FM

InstaNovo-FM is a self-supervised foundation model for tandem mass spectra. Instead of learning to predict a peptide sequence during pretraining, it reconstructs masked regions of a spectrum and turns every spectrum into a reusable 768-dimensional representation. This notebook loads the published encoder, embeds a bundled MS/MS example, demonstrates the mechanics of spectral-library retrieval, and shows how to run the companion *de novo* sequencer.

The tutorial is deliberately small: it runs on the included one-spectrum MGF file and then shows exactly where to substitute an MGF, mzML, mzXML, CSV, parquet file, or a larger `SpectrumDataFrame` for an analysis of your own. A GPU is recommended for the checkpoint downloads and the larger workflows, but the bundled demo also runs on CPU.

## Why embed spectra?

The frozen encoder is useful wherever a spectral representation is more helpful than a task-specific predictor. The accompanying manuscript demonstrates that these embeddings can support:

- **Database-free retrieval and rescue.** Search an embedding index of identified reference spectra, then validate close query–anchor pairs with precursor mass, raw-spectrum similarity, and fragment-ion evidence. This is especially interesting for spectra that a conventional search leaves unassigned.
- **Modification and glycan analysis.** Train a small probe on frozen embeddings instead of retraining the encoder. The manuscript reports strong detection of phosphorylation and glycosylation, and resolves coarse N-glycan families.
- **Run-level quality control and classification.** Aggregate embeddings across a run to characterize experimental conditions without needing peptide or protein identifications.
- **De novo sequencing.** Use the separately released encoder–decoder checkpoint when the desired output is a peptide sequence.

These are research workflows, not identification calls by themselves: use held-out data and an appropriate target–decoy or other validation scheme before transferring any biological label.

## Environment

This notebook is designed to run in [Google Colab](https://colab.research.google.com/) from the published PyPI package — no repository checkout is required. In Colab, select **Runtime → Change runtime type → T4 GPU** before running the cells. CPU also works for this single-spectrum tutorial, but a GPU is much faster for a real library or a *de novo* batch.

The next cell installs the published `instanovo-fm==0.1.0` release and displays pip's download progress. Colab preloads NumPy, while InstaNovo-FM installs a pinned NumPy version; therefore the cell restarts the runtime once after installation. After the restart, run that cell again and then continue. Checkpoints are downloaded automatically from the GitHub release and cached under `~/.cache/instanovo-fm/`.

In [ ]:
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PACKAGE_VERSION = "0.1.0"
is_colab = Path("/content").is_dir()
restart_marker = Path("/content") / f".instanovo_fm_{PACKAGE_VERSION}_ready"

# Colab preloads NumPy. Pip can replace it to satisfy the published package's
# reproducible dependency set, but the already-imported NumPy C extension then
# no longer matches its Python files. Install visibly and restart once before
# importing scientific packages. The marker prevents a restart loop.
needs_colab_bootstrap = is_colab and not restart_marker.exists()
try:
    installed_version = importlib.metadata.version("instanovo-fm")
except importlib.metadata.PackageNotFoundError:
    installed_version = None

requirements = []
if installed_version != PACKAGE_VERSION:
    requirements.append(f"instanovo-fm=={PACKAGE_VERSION}")
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.9")

if needs_colab_bootstrap or requirements:
    print("Installing InstaNovo-FM and its dependencies. This may take a few minutes...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--progress-bar", "on",
        f"instanovo-fm=={PACKAGE_VERSION}", "matplotlib>=3.9",
    ])

if needs_colab_bootstrap:
    restart_marker.touch()
    print("Installation complete. Restarting the Colab runtime; run this cell once more afterwards.")
    os.kill(os.getpid(), 9)

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from instanovo_fm.data import FoundationalDataProcessor
from instanovo_fm.model.encoder import FoundationModel
from instanovo_fm.utils.spectrum_dataframe import SpectrumDataFrame

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}.")

## Load the published foundation encoder

`instanovo-fm-v0.1.0` is the published 89.5M-parameter Thompson-span / isotope-co-masking model. Calling `describe_pretrained` is a convenient way to record exactly which checkpoint was used in an analysis.

In [ ]:
MODEL_ID = "instanovo-fm-v0.1.0"

pd.Series(FoundationModel.describe_pretrained(MODEL_ID), name=MODEL_ID)

In [ ]:
model, model_config = FoundationModel.from_pretrained(MODEL_ID)
model = model.to(device).eval()

print(f"Loaded {MODEL_ID}: {model.n_layers} layers, {model.dim_model}-dimensional embeddings")

## Load and view an MS/MS spectrum

The tutorial downloads a compact annotated MGF spectrum from the InstaNovo GitHub repository, so it does not depend on a local checkout or a large dataset download. Point `SPECTRA_PATH` at your own file to continue with your data; `SpectrumDataFrame.load` also supports mzML, mzXML, CSV, and parquet inputs. Labels are only used below to display the example — the foundation encoder does not need them.

In [ ]:
from urllib.request import urlretrieve

# For your own analysis, replace this with Path("/content/your_spectra.mgf").
SAMPLE_MGF_URL = "https://raw.githubusercontent.com/instadeepai/InstaNovo/main/sample_data/spectra.mgf"
SPECTRA_PATH = Path.cwd() / "instanovo_fm_example.mgf"
if not SPECTRA_PATH.exists():
    urlretrieve(SAMPLE_MGF_URL, SPECTRA_PATH)

sdf = SpectrumDataFrame.load(
    str(SPECTRA_PATH),
    lazy=False,
    is_annotated=True,  # Set False when the file does not contain peptide annotations.
    shuffle=False,
)
raw_dataset = sdf.to_dataset(in_memory=True)

preview_columns = ["scan_number", "sequence", "precursor_mz", "precursor_charge", "retention_time"]
sdf.to_pandas()[[column for column in preview_columns if column in sdf.to_pandas().columns]]

In [ ]:
example = raw_dataset[0]
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(example["mz_array"], 0, example["intensity_array"], color="#1f77b4", linewidth=1)
ax.set(xlabel="m/z", ylabel="intensity", title=f"MS/MS spectrum: {example.get('sequence', 'unannotated')}")
plt.show()

## Preprocess exactly as the checkpoint expects

A model embedding is only meaningful when the input preprocessing matches training. `FoundationalDataProcessor` retains the most intense peaks, square-root transforms and L2-normalizes intensity, normalizes m/z when required by the checkpoint, and pads the result. We disable masking here because this is inference rather than self-supervised training.

In [ ]:
def make_foundation_processor(config):
    return FoundationalDataProcessor(
        n_peaks=config.get("n_peaks", 200),
        min_mz=config.get("min_mz", 50.0),
        max_mz=config.get("max_mz", 2500.0),
        min_intensity=config.get("min_intensity", 1e-6),
        remove_precursor_tol=0.0,
        use_spectrum_utils=config.get("use_spectrum_utils", False),
        normalize_mz=config.get("normalize_mz", True),
        peak_ordering=config.get("peak_ordering", "sorted"),
        annotated=False,
        masking_strategy="none",
    )

processor = make_foundation_processor(model_config)
processed_dataset = processor.process_dataset(raw_dataset)
loader = DataLoader(processed_dataset, batch_size=min(32, len(processed_dataset)), collate_fn=processor.collate_fn)
batch = next(iter(loader))
spectra = batch["spectra"].to(device)

print(f"Model input shape: {tuple(spectra.shape)}  (batch, padded peaks, [m/z, intensity])")

## Turn each spectrum into a learned fingerprint

The encoder converts each MS/MS spectrum into 768 numbers: a **spectral fingerprint**. It is not a list of identified ions, a peptide sequence, or a confidence score. Instead, the fingerprint summarizes peak patterns that the model learned during masked-spectrum reconstruction — for example, fragmentation ladders, isotope and neutral-loss relationships, and broader acquisition context.

For the downstream applications in the manuscript, use `encode_mean_pooled`. It summarizes the final representation of all observed peaks into one vector per spectrum. Spectra with similar fingerprints are candidate neighbours for the same peptide, related chemistry, or similar experimental context; the appropriate interpretation depends on the library and task.

In [ ]:
embeddings = model.encode_mean_pooled(spectra)

print(f"Embedded {len(embeddings)} spectrum/s as {embeddings.shape[1]}-number fingerprints.")
print(f"Fingerprint length: {embeddings[0].norm():.3f} (1.000 is expected).")
print("A length of 1 is only a normalization convention: it makes fingerprints comparable,")
print("not a measure of spectral quality, identification confidence, or biological relevance.")

## A tiny retrieval demo

A real spectral library contains many spectra and known anchor identities. The bundled file has one spectrum, so the following cell makes intensity/dropout perturbations solely to demonstrate the nearest-neighbour API and check that the representation is stable under small changes. It is **not** an identification benchmark.

Because the fingerprints have unit length, their dot product is cosine similarity: **1** means identical fingerprints, values nearer **0** mean less similar directions, and **−1** would mean opposite directions. Treat similarity as a *ranking* signal, not an identification confidence or a universal cutoff. In a real library, compare a candidate to the score distribution of known matches and non-matches, then validate it with raw-spectrum and precursor evidence.

In [ ]:
def perturb_spectrum(spectrum, valid_mask, dropout_fraction, seed):
    """Drop a few non-padding peaks and slightly perturb intensities."""
    generator = torch.Generator(device=spectrum.device).manual_seed(seed)
    result = spectrum.clone()
    valid_indices = torch.where(valid_mask)[0]
    n_drop = int(len(valid_indices) * dropout_fraction)
    if n_drop:
        permutation = torch.randperm(len(valid_indices), generator=generator, device=spectrum.device)
        dropped = valid_indices[permutation[:n_drop]]
        result[dropped] = 0
    noise = 1 + 0.03 * torch.randn(result.shape[0], generator=generator, device=spectrum.device)
    result[:, 1] = (result[:, 1] * noise).clamp_min(0)
    return result

query = spectra[0]
valid = ~batch["spectra_mask"][0].to(device)
library = torch.stack([query] + [perturb_spectrum(query, valid, 0.12, seed) for seed in range(1, 6)])
library_embeddings = model.encode_mean_pooled(library)
similarity = library_embeddings[0] @ library_embeddings.T

pd.DataFrame({
    "library_entry": ["query (exact copy)"] + [f"perturbation {i}" for i in range(1, 6)],
    "cosine_similarity_to_query": similarity.cpu().numpy(),
}).sort_values("cosine_similarity_to_query", ascending=False, ignore_index=True)

### Turn the pattern into a spectral-library search

For a reference library, embed the references once and persist their embeddings with the anchor metadata. For each query batch, calculate the matrix product below and retain the top candidates. In a production workflow, validate candidate label transfers with appropriate controls and evidence from the original spectra; a close embedding alone is insufficient. For larger libraries, replace the matrix product with a FAISS index.

In [ ]:
# reference_embeddings: (n_reference, 768), already L2-normalized
# query_embeddings:     (n_query, 768), already L2-normalized
# reference_metadata:   a DataFrame indexed in the same order as reference_embeddings

def retrieve_top_k(query_embeddings, reference_embeddings, reference_metadata, k=5):
    similarities = query_embeddings @ reference_embeddings.T
    scores, indices = similarities.topk(min(k, len(reference_embeddings)), dim=1)
    rows = []
    for query_id, (query_scores, query_indices) in enumerate(zip(scores, indices, strict=True)):
        for rank, (score, index) in enumerate(zip(query_scores.tolist(), query_indices.tolist(), strict=True), start=1):
            row = {"query_id": query_id, "rank": rank, "embedding_cosine": score}
            row.update(reference_metadata.iloc[index].to_dict())
            rows.append(row)
    return pd.DataFrame(rows)

# Example with the toy library above. Supply real annotation metadata in a real library.
toy_metadata = pd.DataFrame({"library_entry": ["exact copy"] + [f"perturbation {i}" for i in range(1, 6)]})
retrieve_top_k(library_embeddings[:1], library_embeddings, toy_metadata, k=3)

## From spectrum embeddings to a run representation

Run-level analysis is just another pooling step. Embed all spectra in a run in batches, then mean-pool and renormalize their vectors. These run representations can drive visualization, outlier detection, or a simple classifier for known technical labels. Keep experiments separate between training and evaluation when assessing generalization.

In [ ]:
import torch.nn.functional as F

def embed_run(dataset, processor, model, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=processor.collate_fn)
    spectrum_embeddings = []
    for batch in loader:
        spectrum_embeddings.append(model.encode_mean_pooled(batch["spectra"].to(device)))
    return torch.cat(spectrum_embeddings)

# With the bundled sample this is a one-spectrum run; use a complete raw-file
# dataset here for a meaningful run-level representation.
run_spectrum_embeddings = embed_run(processed_dataset, processor, model)
run_embedding = F.normalize(run_spectrum_embeddings.mean(dim=0), dim=0)
print(f"Run representation shape: {tuple(run_embedding.shape)}")

## Optional: de novo peptide sequencing

The foundation encoder itself produces embeddings, not amino-acid sequences. The release also contains a downstream encoder–decoder checkpoint initialized from the foundation model and fine-tuned for *de novo* sequencing. The cell below runs beam search on the same small input. Its prediction is a model hypothesis; apply confidence and FDR controls before using it in an analysis.

In [ ]:
from instanovo.inference import BeamSearchDecoder
from instanovo_fm.downstream.de_novo_sequencing.data import DownstreamDeNovoDataProcessor
from instanovo_fm.downstream.de_novo_sequencing.model import DownstreamDeNovo

DENOVO_MODEL_ID = "instanovo-fm-denovo-v0.1.0"
denovo_model, denovo_config = DownstreamDeNovo.from_pretrained(DENOVO_MODEL_ID)
denovo_model = denovo_model.to(device).eval()

denovo_processor = DownstreamDeNovoDataProcessor(
    residue_set=denovo_model.residue_set,
    n_peaks=denovo_config.get("n_peaks", 200),
    min_mz=denovo_config.get("min_mz", 50.0),
    max_mz=denovo_config.get("max_mz", 2500.0),
    min_intensity=denovo_config.get("min_intensity", 1e-6),
    remove_precursor_tol=denovo_config.get("remove_precursor_tol", 2.0),
    use_spectrum_utils=denovo_config.get("use_spectrum_utils", False),
    normalize_mz=denovo_config.get("normalize_mz", True),
    annotated=False,
    return_str=True,
)
denovo_dataset = denovo_processor.process_dataset(raw_dataset)
denovo_batch = next(iter(DataLoader(denovo_dataset, batch_size=min(16, len(denovo_dataset)), collate_fn=denovo_processor.collate_fn)))

decoder = BeamSearchDecoder(model=denovo_model)
output = decoder.decode(
    spectra=denovo_batch["spectra"].to(device),
    precursors=denovo_batch["precursors"].to(device),
    beam_size=5,
    max_length=denovo_config.get("max_length", 40),
)

sequence_log_probability = output["prediction_log_probability"]
if isinstance(sequence_log_probability, torch.Tensor):
    sequence_log_probability = sequence_log_probability.detach().cpu().numpy()

pd.DataFrame({
    "predicted_peptide": ["".join(tokens) for tokens in output["predictions"]],
    "sequence_log_probability": sequence_log_probability,
})

## Where to go next

- For batch embedding and the paper’s retrieval, linear-probe, clustering, attribution, and UMAP tasks, start with `uv run python -m instanovo_fm.eval.embed_evaluation --config-name foundational_local`.
- For end-to-end de novo prediction over MGF, mzML, mzXML, CSV, or parquet data, use `instanovo-fm denovo predict` and the options documented in `src/instanovo_fm/downstream/de_novo_sequencing/README.md`.
- The manuscript, [Learning from tandem mass spectra at scale with a self-supervised foundation model for proteomics](https://doi.org/10.64898/2026.09.03.747733), describes the masked-reconstruction objective and the reported use cases in detail.

The most productive next experiment is usually modest: embed a held-out collection from your own workflow, inspect its nearest neighbours and low-dimensional structure, then add the smallest appropriately split probe needed to test a concrete biological or technical hypothesis.